In [3]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

from evaluation.eval_utils import *
import numpy as np
import faiss
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from generation.cohere_generation import *

import json
from openai import OpenAI
from string import Template
from collections import defaultdict

client = OpenAI(
    api_key="sk-2367b265559a4ae6b607bff8755ef431",
    base_url="https://api.deepseek.com",
)

dotenv_path: d:\NLP_Project\NLP_project\key.env


In [8]:
#########################################
# Retrieval Method Functions
#########################################

#########################################
# Sparse Retrieval Methods - TF-IDF 
#########################################
def tfidf_retrieval(query, config):
    """
    TF-IDF retrieval.
    config must include:
      - "vectorizer": a fitted TfidfVectorizer,
      - "doc_matrix": document-term matrix,
      - "passages": list of passages,
      - "chunk_ids": list of passage identifiers,
      - "top_k": number of top results.
    """
    vectorizer = config["vectorizer"]
    doc_matrix = config["doc_matrix"]
    passages = config["passages"]
    chunk_ids = config["chunk_ids"]
    top_k = config.get("top_k", 5)
 
    if isinstance(query, list):
        for i in query:
            query_vec = vectorizer.transform([i])
            cosine_similarities = (doc_matrix @ query_vec.T).toarray().flatten()
            sorted_indices = np.argsort(cosine_similarities)[::-1][:top_k]
            
            results = [{"chunk_id": chunk_ids[j],
                        "score": float(cosine_similarities[j]),
                        "sub_query": i,
                        "passage": passages[j],}
                    for j in sorted_indices]
    else:
        query_vec = vectorizer.transform([query])
        cosine_similarities = (doc_matrix @ query_vec.T).toarray().flatten()
        sorted_indices = np.argsort(cosine_similarities)[::-1][:top_k]
        
        results = [{"chunk_id": chunk_ids[j],
                    "score": float(cosine_similarities[j]),
                    "sub_query": query[0],
                    "passage": passages[j],            }
                for j in sorted_indices]
 
    return results

#########################################
# Sparse Retrieval Methods - BM25
#########################################
def bm25_retrieval(query, config):
    """
    BM25 retrieval.
    config must include:
      - "bm25": a BM25Okapi object,
      - "passages": list of passages,
      - "chunk_ids": list of passage identifiers,
      - "top_k": number of top results.
    """
    bm25 = config["bm25"]
    passages = config["passages"]
    chunk_ids = config["chunk_ids"]
    top_k = config.get("top_k", 5)

    if isinstance(query, list):
        for i in query:
            tokenized_query = word_tokenize(i)
            scores = bm25.get_scores(tokenized_query)
            sorted_indices = np.argsort(scores)[::-1][:top_k]
            
            results = [{"chunk_id": chunk_ids[j],
                        "score": float(scores[j]),
                        "sub_query": i,
                        "passage": passages[j],}
                    for j in sorted_indices]
    else:
        tokenized_query = word_tokenize(query)
        scores = bm25.get_scores(tokenized_query)
        sorted_indices = np.argsort(scores)[::-1][:top_k]
        
        results = [{"chunk_id": chunk_ids[i],
                    "score": float(scores[i]),
                    "sub_query": query[0],
                    "passage": passages[i],}
                for i in sorted_indices]    
        
    return results

#########################################
# Dense Retrieval Methods 
#########################################

def dense_retrieval_subqueries(queries, config):

    if isinstance(queries, str):
        queries = [queries]

    results = []
    top_k = config.get("top_k", 5)
    for query in queries:
        query_emb = query_embed_search(query, config["all_subqueries"], config["subquery_index"])
        query_emb = query_emb.reshape(1, -1)  
        distances, indices = config["faiss_index"].search(query_emb, top_k)
        results.extend([
            {
                "sub_query": query,
                "chunk_id": config["chunk_ids"][i],
                "passage": config["passages"][i],
                "score": float(distances[0][j])
            } for j, i in enumerate(indices[0])
        ])
    return results

def query_embed_search(query, all_queries_list, index):
    """
    Given a query, finds its embedding from the precomputed subqueries index.
    """
    try:
        position = all_queries_list.index(query)
        return index.reconstruct(position)
    except ValueError:
        raise ValueError(f"Query '{query}' not found in the list of all queries.")
    
def hybrid(query, config):
 
    # Check if the query is a list with at least two elements
    if isinstance(query, list) and len(query) >= 2:
        query_sparse = query[0]
        query_dense = query[1]
    else:
        print("Query should be a list of two queries: [sparse_query, dense_query].")
 
    intermediate_k = config.get("intermediate_k")
    final_k = config.get("final_k")
    mode = config.get("mode", "intersection")
    order = config.get("order", "sparse_dense")
    # note for order: "sparse_dense" means sparse retrieval first, then dense retrieval.
    sparse_config = config["sparse_config"].copy()
    dense_config = config["dense_config"].copy()
    # set sparse_config as subquery retrieval config
    
    if order == "dense_sparse":
        # intermediate k is for dense
        dense_config["top_k"] = intermediate_k
        sparse_config["query_type"] = "sub_questions"
    elif order == "sparse_dense":
        # intermediate k is for sparse
        sparse_config["top_k"] = intermediate_k
 
    # Step 1: Sparse retrieval using query_sparse.
    sparse_results = config["sparse_func"](query_sparse, sparse_config)
    
    # Step 2: Dense retrieval using query_dense.
    
    dense_results = config["dense_func"](query_dense, dense_config)
    
    # Step 3: Combine candidate sets.
    if mode == "intersection":
        if order == "sparse_dense":
            candidate_ids = set(r["chunk_id"] for r in sparse_results)
            filtered_dense = [r for r in dense_results if r["chunk_id"] in candidate_ids]
            if not filtered_dense:
                filtered_dense = dense_results
        elif order == "dense_sparse":
            candidate_ids = set(r["chunk_id"] for r in dense_results)
            filtered_dense = [r for r in sparse_results if r["chunk_id"] in candidate_ids]
            if not filtered_dense:
                filtered_dense = sparse_results
        else:
            raise ValueError("Invalid order specified.")
    # if mode == "intersection":
    #     candidate_ids = set(r["chunk_id"] for r in sparse_results)
    #     filtered_dense = [r for r in dense_results if r["chunk_id"] in candidate_ids]
    #     if not filtered_dense:
    #         filtered_dense = dense_results  # fallback if intersection is empty
    elif mode == "union":
        candidate_ids = set(r["chunk_id"] for r in sparse_results).union(
                        set(r["chunk_id"] for r in dense_results))
        dense_dict = {r["chunk_id"]: r for r in dense_results}
        filtered_dense = []
        for cid in candidate_ids:
            if cid in dense_dict:
                filtered_dense.append(dense_dict[cid])
            else:
                filtered_dense.append({"chunk_id": cid, "score": 0.0})
    else:
        raise ValueError("Invalid mode. Choose 'intersection' or 'union'.")
    
    # Step 4: Sort by dense score (descending) and select top final_k.
    sorted_dense = sorted(filtered_dense, key=lambda x: x["score"], reverse=True)
    #final_results = sorted_dense[:final_k]
    final_results = sorted_dense
    
    # ***** String Constraint Enforcement After Hybrid Retrieval *****
    # Ensure that query_dense is a string.
    query_dense = str(query_dense)
    # For each candidate, ensure the passage is a string.
    for candidate in final_results:
        candidate["passage"] = str(candidate.get("passage", ""))
    
    # Step 5 (Optional): re-ranking.

    if config.get("rerank") in config:
        
        sparse_dict = {r["chunk_id"]: r["score"] for r in sparse_results}
        dense_dict = {r["chunk_id"]: r["score"] for r in dense_results}
 
        all_ids = list(set(sparse_dict.keys()).union(set(dense_dict.keys())))
        sparse_scores = np.array([sparse_dict.get(cid, 0.0) for cid in all_ids])
        dense_scores = np.array([dense_dict.get(cid, 0.0) for cid in all_ids])
 
        # Avoid divide-by-zero in normalization
        if np.max(sparse_scores) > 0:
            sparse_scores /= np.max(sparse_scores)
        if np.max(dense_scores) > 0:
            dense_scores /= np.max(dense_scores)
 
        alpha = 0.3
        combined_scores = alpha * sparse_scores + (1 - alpha) * dense_scores
        
        # Build final result list
        final_results = []
        for i, cid in enumerate(all_ids):
            result = {
                "chunk_id": cid,
                "score": float(combined_scores[i]),
                "passage": str(next((r["passage"] for r in dense_results if r["chunk_id"] == cid), "")),
            }
            final_results.append(result)
 
        # Sort by hybrid score
        final_results = sorted(final_results, key=lambda x: x["score"], reverse=True)
 
    
    
    return final_results[:final_k]

#########################################
# Knowledge Graph Retrieval Method Functions
#########################################

def process_gpt(query):
    
    system_prompt = """
    You are an expert in entity extraction.
    """
    
    with open('../data/HP_KG_5_chunks/Node.json', 'r') as file:
        node_data = json.load(file)
    
    with open('../data/HP_KG_5_chunks/Special.json', 'r') as file:
        magic_data = json.load(file)

    all_entities = {item['name']: item['id'] for item in node_data+magic_data}
    entity_names = list(all_entities.keys())
    
    prompt_template = Template("""
        You are an expert in entity recognition.

        DO NOT answer the question — only pull out names that appear in the query from the provided list.

        Return a Python list of matching full names (exact or partial matches) — nothing else.
    
        Given a query, identify any matching names from the list below. Matching should be:
        - Match should be case-insensitive.
        - If the query says "Harry", and "Harry Potter" is in the entity list, include "Harry Potter".
        - If the query says "Professor [LastName]", look for any full name in the entity list that ends with [LastName], and return the full name instead.
        - You MUST include spell and potion names if they are mentioned
        - Return only names that are exactly in the entity list.
        - Return only a valid Python list of matched names, like ["Harry Potter", "Sirius Black"]

        Entities:
        $entity_names

        Query:
        "$query"
    """)
    
    user_prompt = prompt_template.substitute(
        entity_names="\n".join(entity_names),
        query=query
    ) 
    
    completion = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    extracted_names = json.loads(completion.choices[0].message.content.strip())
    print(extracted_names)
    
    matched_ids = [all_entities[name] for name in extracted_names if name in all_entities]
    return matched_ids

def find_chunk_id(target_ids):
    with open("../data/Node_Dictionary.json", "r") as f1:
        node_dict = json.load(f1)

    with open("../data/Special_Dictionary.json", "r") as f2:
        spec_dict = json.load(f2)
    
    full_dict = node_dict.copy()
    full_dict.update(spec_dict)
    
    scene_counts = defaultdict(int)
    num_entities = 0     
    for eid in target_ids:
        entity = full_dict.get(eid)
        if entity:
            appears = entity.get("list of appear", [])
            for scene_id in appears:
                scene_counts[int(scene_id)] += 1
            num_entities += 1
    shared_scenes = sorted([scene_id for scene_id, count in scene_counts.items() if count == num_entities])

    return shared_scenes
def KG_dense_retrieval(queries, config):
    if isinstance(queries, str):
        queries = [queries]

    results = []
    kg_ids = []
    relation_to_kgid_map = config["relation_to_kgid_map"]
    dense_config = config["dense_config"]
    for query in queries:
        #query_emb = model.encode(query, convert_to_numpy=True, normalize_embeddings=True)
        query_emb = query_embed_search(query, config["all_subqueries"], config["subquery_index"])
        query_emb = query_emb.reshape(1, -1)  
        distances, indices = config["faiss_index"].search(query_emb, 1000)
        entities = process_gpt(query)
        shared_kg = find_chunk_id(entities)
        relation_rank_index = indices[0]
        for i in relation_rank_index:
            if relation_to_kgid_map[i] in shared_kg:
                kg_ids.append(relation_to_kgid_map[i])
        if len(kg_ids) == 0:
            print('KG method failed')
            print('return the dense retrieval')
            return dense_retrieval_subqueries(queries, dense_config)
        else:
            chunk_id_list = []
            for kg_id in kg_ids:
                chunk_id_list.extend([i for i in range(5*kg_id, 5*kg_id+5)])
                chunk_id_list = list(set(chunk_id_list))
            sel = faiss.IDSelectorArray(chunk_id_list)
            params = faiss.SearchParameters()
            params.sel = sel
            return dense_retrieval_subqueries_for_finetune(queries, config["all_subqueries"], config["subquery_index"],  config["faiss_index"], config["chunk_ids"], config["passages"], top_k=5,params=params)
        
    
def dense_retrieval_subqueries_for_finetune(queries, all_queries_list, sub_queries_index, faiss_index, all_chucks, all_passages, top_k=5,params = faiss.SearchParameters()):
    if isinstance(queries, str):
        queries = [queries]

    results = []
    for query in queries:
        #query_emb = model.encode(query, convert_to_numpy=True, normalize_embeddings=True)
        query_emb = query_embed_search(query, all_queries_list, sub_queries_index)
        query_emb = query_emb.reshape(1, -1)  # Reshapes to (1, d)
        distances, indices = faiss_index.search(query_emb, top_k, params=params)
        #
        results.extend([
            {
                "sub_query": query,
                "chunk_id": all_chucks[i],
                "passage": all_passages[i],
                "score": float(distances[0][j])
            } for j, i in enumerate(indices[0])
        ])
    return results

#########################################
# Dataset Configuration
#########################################

# Global corpus (for all retrieval methods that use it).
def load_corpus(corpus_file):
    with open(corpus_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    passages = [entry["passage"] for entry in data if "passage" in entry]
    chunk_ids = [entry["chunk_id"] for entry in data if "chunk_id" in entry]
    print(f"Loaded {len(passages)} corpus passages")
    return passages, chunk_ids

# A helper function to load subqueries from a ground truth file.
def retrieve_all_subqueries(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        qa_data = json.load(f)
    subqueries = []
    for item in qa_data:
        subqueries.extend(item["sub_questions"])
    return subqueries

#########################################################
# PATHES
#########################################################

# Paths for the corpus and QA sets.
CORPUS_FILE = "../data/chunked_text_all_together_cleaned.json"
QA_PATH = "../data/QA_set"
QA_EMBEDDED_PATH = "../embedding/BAAI/bge-base-en-v1.5"
PM_PATH = "../performance"
corpus_dense_index = faiss.read_index("../embedding/BAAI/bge-base-en-v1.5/hp_all_bge.index")


###################################################
# Essential Data for Retrieval
###################################################
passages, chunk_ids = load_corpus(CORPUS_FILE)

# Build TF-IDF index for the corpus.
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
doc_matrix = tfidf_vectorizer.fit_transform(passages)

# Build BM25 index for the corpus.
tokenized_passages = [word_tokenize(p.lower()) for p in passages]
bm25 = BM25Okapi(tokenized_passages)

# Load the relation to KG ID mapping.
with open("../data/relation_to_kgid_map.json", "r", encoding="utf-8") as f:
    relation_to_kgid_map = json.load(f)
###################################################
# Configurations for Retrieval Methods
###################################################
datasets = [
    {
        "name": "easy_single",
        "gt_path": os.path.join(QA_PATH, "easy_single_labeled.json"),
        "dense": {
            "subqueries": retrieve_all_subqueries(os.path.join(QA_PATH, "easy_single_labeled.json")),
            "subquery_index": faiss.read_index(os.path.join(QA_EMBEDDED_PATH, "easy_single_labeled_embeddings.index"))
        }
    },
    {
        "name": "medium_single",
        "gt_path": os.path.join(QA_PATH, "medium_single_labeled.json"),
        "dense": {
            "subqueries": retrieve_all_subqueries(os.path.join(QA_PATH, "medium_single_labeled.json")),
            "subquery_index": faiss.read_index(os.path.join(QA_EMBEDDED_PATH, "medium_single_labeled_embeddings.index"))
        }
    },
    {
        "name": "medium_multi",
        "gt_path": os.path.join(QA_PATH, "medium_multi_labeled.json"),
        "dense": {
            "subqueries": retrieve_all_subqueries(os.path.join(QA_PATH, "medium_multi_labeled.json")),
            "subquery_index": faiss.read_index(os.path.join(QA_EMBEDDED_PATH, "medium_multi_labeled_embeddings.index"))
        }
    },
    {
        "name": "hard_single",
        "gt_path": os.path.join(QA_PATH, "hard_single_labeled.json"),
        "dense": {
            "subqueries": retrieve_all_subqueries(os.path.join(QA_PATH, "hard_single_labeled.json")),
            "subquery_index": faiss.read_index(os.path.join(QA_EMBEDDED_PATH, "hard_single_labeled_embeddings.index"))
        }
    },
    {
        "name": "hard_multi",
        "gt_path": os.path.join(QA_PATH, "hard_multi_labeled.json"),
        "dense": {
            "subqueries": retrieve_all_subqueries(os.path.join(QA_PATH, "hard_multi_labeled.json")),
            "subquery_index": faiss.read_index(os.path.join(QA_EMBEDDED_PATH, "hard_multi_labeled_embeddings.index"))
        }
    }
]


# Create a dictionary mapping retrieval method names to their functions.
retrieval_methods = {
    "tfidf": tfidf_retrieval,
    "bm25": bm25_retrieval,
    "dense": dense_retrieval_subqueries,
    "tfidf_dense": hybrid,
    "bm25_dense": hybrid,
    "dense_tfidf": hybrid,
    "dense_bm25": hybrid,
    "dense_tfidf_parallel": hybrid,
    "dense_bm25_parallel": hybrid,
    "KG_dense": KG_dense_retrieval

    
}

# For each dataset, define a configuration for each retrieval method.
# Here we create a dictionary keyed by retrieval method for each dataset.
for ds in datasets:
    ds.setdefault("retrieval_config", {})
    #TF-IDF configuration.
    ds["retrieval_config"]["tfidf"] = {
        "passages": passages,
        "chunk_ids": chunk_ids,
        "vectorizer": tfidf_vectorizer,
        "doc_matrix": doc_matrix,
        "top_k": 20,
        "query_type": "sub_questions"  # or "sub_questions" based on your data structure
    }
    # BM25 configuration.
    ds["retrieval_config"]["bm25"] = {
        "passages": passages,
        "chunk_ids": chunk_ids,
        "bm25": bm25,
        "top_k": 20,
        "query_type": "sub_questions"  # or "sub_questions" based on your data structure
    }
    
    #Dense retrieval configuration.
    ds["retrieval_config"]["dense"] = {
        "passages": passages,
        "chunk_ids": chunk_ids,
        "all_subqueries": ds["dense"]["subqueries"],
        "subquery_index": ds["dense"]["subquery_index"],
        "faiss_index": corpus_dense_index,
        "top_k": 5,
        "query_type": "sub_questions"  # or "question" based on your data structure
    }
        # --- Add Hybrid Configurations ---
    # Hybrid TF-IDF + Dense configuration.
    ds["retrieval_config"]["tfidf_dense"] = {
        "sparse_func": tfidf_retrieval,
        "sparse_config": ds["retrieval_config"]["tfidf"],
        "dense_func": dense_retrieval_subqueries,
        "dense_config": ds["retrieval_config"]["dense"],
        "intermediate_k": 1000,   # You can choose this value independently.
        "final_k": 20,           # And choose the final number of results.
        "mode": "intersection" , # or "union"
        "query_type": "combine",  # or "sub_questions" based on your data structure
        "order": "sparse_dense"  # or "dense_sparse"

    }
    # Hybrid BM25 + Dense configuration.
    ds["retrieval_config"]["bm25_dense"] = {
        "sparse_func": bm25_retrieval,
        "sparse_config": ds["retrieval_config"]["bm25"],
        "dense_func": dense_retrieval_subqueries,
        "dense_config": ds["retrieval_config"]["dense"],
        "intermediate_k": 1000,
        "final_k": 20,
        "mode": "intersection" , # or "union"
        "query_type": "combine",  # or "sub_questions" based on your data structure
        "order": "sparse_dense"  # or "dense_sparse"
    }
    ds["retrieval_config"]["dense_tfidf"] = {
        "sparse_func": tfidf_retrieval,
        "sparse_config": ds["retrieval_config"]["tfidf"],
        "dense_func": dense_retrieval_subqueries,
        "dense_config": ds["retrieval_config"]["dense"],
        "intermediate_k": 1000,   # You can choose this value independently.
        "final_k": 20,           # And choose the final number of results.
        "mode": "intersection" , # or "union"
        "query_type": "combine",  # or "sub_questions" based on your data structure
        "order": "dense_sparse"  # or "dense_sparse"

    }
    # Hybrid BM25 + Dense configuration.
    ds["retrieval_config"]["dense_bm25"] = {
        "sparse_func": bm25_retrieval,
        "sparse_config": ds["retrieval_config"]["bm25"],
        "dense_func": dense_retrieval_subqueries,
        "dense_config": ds["retrieval_config"]["dense"],
        "intermediate_k": 1000,
        "final_k": 20,
        "mode": "intersection" , # or "union"
        "query_type": "combine",  # or "sub_questions" based on your data structure
        "order": "dense_sparse"  # or "dense_sparse"
    }

    ds["retrieval_config"]["dense_tfidf_parallel"] = {
        "sparse_func": tfidf_retrieval,
        "sparse_config": ds["retrieval_config"]["tfidf"],
        "dense_func": dense_retrieval_subqueries,
        "dense_config": ds["retrieval_config"]["dense"],
        "intermediate_k": 1000,   # You can choose this value independently.
        "final_k": 20,           # And choose the final number of results.
        "mode": "union" , # or "union"
        "query_type": "combine",  # or "sub_questions" based on your data structure
        "order": "dense_sparse",  # or "dense_sparse"
        "rerank": True

    }
    # Hybrid BM25 + Dense configuration.
    ds["retrieval_config"]["dense_bm25_parallel"] = {
        "sparse_func": bm25_retrieval,
        "sparse_config": ds["retrieval_config"]["bm25"],
        "dense_func": dense_retrieval_subqueries,
        "dense_config": ds["retrieval_config"]["dense"],
        "intermediate_k": 1000,
        "final_k": 20,
        "mode": "union" , # or "union"
        "query_type": "combine",  # or "sub_questions" based on your data structure
        "order": "dense_sparse",
        "rerank": True # or "dense_sparse"
    }
    # KG + Dense configuration.
    ds["retrieval_config"]["KG_dense"] = {
        "all_subqueries": ds["dense"]["subqueries"],
        "subquery_index": ds["dense"]["subquery_index"],
        "faiss_index": corpus_dense_index,
        "top_k": 5,
        "query_type": "sub_questions",
        "relation_to_kgid_map":  relation_to_kgid_map,
        "dense_config": ds["retrieval_config"]["dense"],
        "chunk_ids": chunk_ids,
        "passages": passages
        }

Loaded 9251 corpus passages


In [ ]:


########################
# Evaluation of the generation model.
########################

generator = CohereGenerator(model="command-r")
#select_datasets = [ "medium_single"]
# name of dataset to be evaluated.
#select_datasets = [ "easy_single", "medium_single", "medium_multi", "hard_single", "hard_multi"]
select_retrieval_methods = ["KG_dense"]
###################################################
# Tested models: "tfidf_dense", "bm25_dense", "dense_tfidf", "dense_bm25", "dense"
# Filter datasets based on the selected names.
#datasets = [ds for ds in datasets if ds["name"] in select_datasets]
# Filter retrieval methods based on the selected names.
retrieval_methods = {k: v for k, v in retrieval_methods.items() if k in select_retrieval_methods}
PM_PATH = "../performance"
# save the results to a JSON file.
save_path = os.path.join(PM_PATH, "generation_performance_results_eval.json")

#results = []  # This list will hold the overall performance for each dataset/model combination.

with open(save_path, "r") as f:
    results = json.load(f)
# check the selected method name is in the results, if it is, remove it.
# for all datasets, if the method name is in the results, remove it.
# for ds in datasets:
#     for method_name in retrieval_methods.keys():
#         for result in results:
#             if ds["name"] == result["data_set"] and method_name == result["generation_model"]:
#                 results.remove(result)
# Loop over each dataset.
for ds in datasets:
    # Load ground truth data.
    with open(ds["gt_path"], "r", encoding="utf-8") as f:
        ground_truth_data = json.load(f)
    print(f"Dataset '{ds['name']}': loaded {len(ground_truth_data)} ground truth examples.")

    
    # Loop over each generation model.
    for method_name, method_func in retrieval_methods.items():
        print(f"Evaluating dataset '{ds['name']}' with generation model '{method_name}'...")
        # Check if a configuration exists for this method.
        if method_name in ds["retrieval_config"]:
            config = ds["retrieval_config"][method_name]
            predictions = []
            references = []
            test_results = []  # Store detailed results for each example.

            # Process each example in the ground truth.
            for example in ground_truth_data:
                origin_question = example["question_variants"]
                subquestions = example["sub_questions"]
                # For hybrid retrieval, we assume the query is a two-element list:
                # [query_for_sparse, query_for_dense]
                query = [origin_question, subquestions]
                query_type = config.get("query_type")
                if query_type == "sub_questions":
                    question = query[1]
                elif query_type == "question":
                    question = query[0]
                else:
                    question = query 
                # Retrieve candidates using the model-specific retrieval function.
                retrieval_results = method_func(question, config)
                
                
                # Generate an answer using the generation function.
                # generation_answer is expected to return previous Q&A and a final answer.
                previous_qa, final_answer = generator.generation_answer(
                    origin_question, subquestions, retrieval_results, top_k=5, max_tokens=60
                )
                
                predictions.append(final_answer)
                # Assume the ground truth contains a list of reference answers under the key "list of reference".
                references.append(example["list of reference"])
                
                # Save the details for later inspection.
                test_results.append({
                    "question": origin_question,
                    "sub_questions": subquestions,
                    "retrieval_results": retrieval_results,
                    "previous_qa": previous_qa,
                    "final_answer": final_answer,
                    "gt_answer": example['answer'],
                    "references": example["list of reference"],
                    "dataset": ds["name"],
                    "generation_model": method_name,
                })
            # Process the references to extract passage texts.
            # passages = []
            # for ref in references:
            #     p = []
            #     for r in ref:
            #         p.append(r["passage"])
            #     passages.append(p)
            predictions = [result["final_answer"] for result in test_results]
            gt_answers = [result["gt_answer"] for result in test_results]
            questions = [result['question'] for result in test_results]
            # Compute generation evaluation metrics.
            gen_metrics = MetricCollection({
                    "sas": SASScore(),
                    "GPT Score": CohereGPTScore(),
                    "bertscore": BERTScore(model_name_or_path="bert-base-uncased", batch_size=16)
                })
            gen_metrics.update(predictions, gt_answers, questions, metric_type="generation")
            gen_results = gen_metrics.compute(metric_type="generation")
            
            print(f"\nGeneration evaluation metrics for dataset '{ds['name']}' with generation model '{method_name}':")
            print(gen_results)
            
            # Append the overall performance for this dataset and generation model.
            results.append({
                "data_set": ds["name"],
                "generation_model": method_name,
                "performance": gen_results
            })

Cohere API key loaded from key.env
Dataset 'easy_single': loaded 253 ground truth examples.
Evaluating dataset 'easy_single' with generation model 'tfidf'...

Generation evaluation metrics for dataset 'easy_single' with generation model 'tfidf':
{'sas': {'mean_similarity': 0.38385869658703037}, 'GPT Score': {'average GPTscore': 0.5104743083003955}, 'bertscore': {'precision': 0.3580032289028168, 'recall': 0.5329217910766602, 'f1': 0.42288801074028015}}
Evaluating dataset 'easy_single' with generation model 'bm25'...

Generation evaluation metrics for dataset 'easy_single' with generation model 'bm25':
{'sas': {'mean_similarity': 0.3730606654946041}, 'GPT Score': {'average GPTscore': 0.5035573122529645}, 'bertscore': {'precision': 0.35288530588150024, 'recall': 0.5315847396850586, 'f1': 0.4190252125263214}}
Evaluating dataset 'easy_single' with generation model 'dense'...

Generation evaluation metrics for dataset 'easy_single' with generation model 'dense':
{'sas': {'mean_similarity': 0

In [10]:
PM_PATH = "../performance"
# save the results to a JSON file.
save_path = os.path.join(PM_PATH, "generation_performance_results_first_try.json")
# Ensure the directory exists.
os.makedirs(os.path.dirname(save_path), exist_ok=True)
# Save the results to the specified JSON file.
with open(save_path, "w") as f:
    json.dump(results, f, indent=2)
# Print a message indicating where the results were saved.
print(f"Results saved to {save_path}")

Results saved to ../performance\generation_performance_results_first_try.json
